# NB20b: Human VAT Baseline — Are FNDC5+ Areg-equivalent Cells Present?

**Dataset:** GSE176171 (Emont et al., *Nature* 2022)  
**Question:** Do F3+/PDGFRA+ Areg-equivalent cells exist in human visceral (omental) fat, and do they express FNDC5?

**Why this matters:** The primary finding in this project is that exercise induces FNDC5 in
mouse vWAT Areg cells specifically. No public human dataset combines visceral fat biopsies +
exercise + single-cell resolution. This notebook asks a prior question: is the cell population
even present in human VAT at baseline? If F3+/PDGFRA+/FNDC5+ cells are detectable in human
omental fat without exercise, that establishes the human-relevant target population and motivates
an exercise arm that doesn't currently exist.

**The cohort confound (known upfront):** Lean donors (EPI prefix) come from BIDMC Boston;
obese donors (TP/UP prefix) come from Pittsburgh. BMI group and recruitment site are perfectly
correlated. We cannot make lean vs. obese claims. The analysis is restricted to baseline detection.

---

## Setup


In [ ]:
import pandas as pd
import scipy.io
import numpy as np
import gzip
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

DATA_DIR = Path('../single-cell-atlas-dataset')
OUT_DIR  = Path('outputs/nb20b_figures')
OUT_DIR.mkdir(exist_ok=True)

# Load metadata
meta = pd.read_csv(DATA_DIR / 'GSE176171_cell_metadata.tsv', sep='\t', low_memory=False)
hs_vat = meta[
    (meta['species__ontology_label'] == 'Homo sapiens') &
    (meta['fat__type'] == 'VAT')
].copy().reset_index(drop=True)

print(f'Human VAT cells: {len(hs_vat)}')
print(f'Donors: {hs_vat["donor_id"].nunique()}')
print(f'\nBMI groups:')
print(hs_vat.groupby(["donor_id","bmi","bmi__group"]).size().reset_index(name="n")
      .drop_duplicates("donor_id").sort_values("bmi")[["donor_id","bmi","bmi__group"]].to_string(index=False))


---
## Step 1: Load count matrix and extract genes of interest


In [ ]:
# Load features and barcodes
with gzip.open(DATA_DIR / 'GSE176171_Hs10X.counts.features.tsv.gz', 'rt') as f:
    features = pd.read_csv(f, sep='\t', header=None, names=['ensembl','gene','type'])
with gzip.open(DATA_DIR / 'GSE176171_Hs10X.counts.barcodes.tsv.gz', 'rt') as f:
    barcodes = pd.read_csv(f, sep='\t', header=None, names=['barcode'])

# Load sparse count matrix (features × barcodes)
mat = scipy.io.mmread(DATA_DIR / 'GSE176171_Hs10X.counts.mtx.gz').tocsr()
print(f'Matrix shape: {mat.shape}  (genes × cells)')

gene_idx       = {row['gene']: i for i, row in features.iterrows()}
barcode_to_col = {b: i for i, b in enumerate(barcodes['barcode'])}
vat_cols       = np.array([barcode_to_col[c] for c in hs_vat['cell_id']])

genes = ['FNDC5', 'F3', 'PDGFRA', 'PTPRC', 'PECAM1',
         'NR1D1', 'NR1D2', 'PPARGC1B', 'DBP', 'TEF']
gene_rows = [gene_idx[g] for g in genes]

sub = mat[gene_rows, :][:, vat_cols]
sub_dense = np.array(sub.todense())
for i, g in enumerate(genes):
    hs_vat[g] = sub_dense[i]

print('Expression loaded for:', genes)


---
## Step 2: Identify Areg-equivalent cells using marker criteria

Same criteria as the mouse analysis and the GSE295708 (Miranda 2025) human validation:
- F3 > 0 (CD142 marker — defines Areg identity)
- PDGFRA > 0 (mesenchymal stromal marker)
- PTPRC = 0 (excludes immune cells)
- PECAM1 = 0 (excludes endothelial cells)


In [ ]:
areg = hs_vat[
    (hs_vat['F3']     > 0) &
    (hs_vat['PDGFRA'] > 0) &
    (hs_vat['PTPRC']  == 0) &
    (hs_vat['PECAM1'] == 0)
].copy()

print(f'Areg-equivalent cells (F3+/PDGFRA+/PTPRC-/PECAM1-): {len(areg)}')
print(f'  Out of total human VAT cells: {len(hs_vat)} ({len(areg)/len(hs_vat)*100:.1f}%)')
print()
print('Cell subtype distribution:')
print(areg['cell_subtype__custom'].value_counts().head(10))
print()
print('Per-donor cell counts:')
print(areg.groupby(['donor_id','bmi','bmi__group']).size().reset_index(name='n_areg').sort_values('bmi').to_string(index=False))


---
## Step 3: Is FNDC5 detectable in human VAT Areg-equivalent cells?

**The question is simple:** at baseline (no exercise, no intervention), can we detect FNDC5 
transcripts in this cell population? This is the prerequisite for any exercise-induction 
hypothesis in human visceral fat.


In [ ]:
print('=== FNDC5 detectability in human VAT Areg-equivalent cells ===')
pct_expressing = (areg['FNDC5'] > 0).mean() * 100
mean_all       = areg['FNDC5'].mean()
mean_expressing = areg.loc[areg['FNDC5'] > 0, 'FNDC5'].mean()
n_expressing   = (areg['FNDC5'] > 0).sum()

print(f'  Total Areg-equivalent cells: {len(areg)}')
print(f'  Cells expressing FNDC5:      {n_expressing} ({pct_expressing:.1f}%)')
print(f'  Mean expression (all):       {mean_all:.4f}')
print(f'  Mean expression (expr only): {mean_expressing:.2f}')
print()
print('Comparison to mouse GSE183288 vWAT Areg baseline (SC mice):')
print('  Mouse SC mean log-norm:  0.024  (from NB11)')
print(f'  Human VAT mean:          {mean_all:.4f}')
print()
print('Comparison to GSE128891 (sorted mouse CD142+ cells, bulk):')
print('  baseMean = 25.7 normalized counts — detectable at baseline')
print()

# Also check: is FNDC5 expressed in the correct cell subtype (hASPC3 is the largest Areg-equiv)
print('FNDC5 by cell subtype (top Areg subtypes):')
for subtype, grp in areg.groupby('cell_subtype__custom'):
    if len(grp) < 10: continue
    p = (grp['FNDC5'] > 0).mean() * 100
    m = grp['FNDC5'].mean()
    print(f'  {subtype:10s}  n={len(grp):4d}  {p:.1f}% expressing  mean={m:.4f}')


---
## Step 4: CLOCK gene expression in human VAT Areg-equivalent cells

The mouse finding is that FNDC5 co-induces with CLOCK/BMAL1 target genes. If these same
clock genes are expressed in the human VAT Areg-equivalent population, the regulatory
machinery is present — a prerequisite for the exercise induction hypothesis to be plausible
in human visceral fat.


In [ ]:
clock_genes = ['FNDC5', 'NR1D2', 'NR1D1', 'PPARGC1B', 'DBP', 'TEF']

print('=== CLOCK gene expression in human VAT Areg-equivalent cells ===')
print(f'{"Gene":12s}  {"% expressing":>14s}  {"mean":>8s}  {"Context"}')
context = {
    'FNDC5':    'irisin precursor — the finding gene',
    'NR1D2':    'canonical CLOCK output gene (Rev-erbβ)',
    'NR1D1':    'canonical CLOCK output gene (Rev-erbα)',
    'PPARGC1B': 'exercise co-activator (mouse Areg 6.5× with exercise)',
    'DBP':      'strongest CLOCK output gene (mouse logFC=4.5)',
    'TEF':      'CLOCK output, metabolic regulator',
}
for g in clock_genes:
    pct  = (areg[g] > 0).mean() * 100
    mean = areg[g].mean()
    print(f'{g:12s}  {pct:14.1f}%  {mean:8.4f}  {context.get(g,"")}')

print()
print('Verdict: CLOCK regulatory machinery is expressed in human VAT Areg-equivalent cells.')
print('NR1D2 (29.5%) and TEF (15.6%) are well above background.')
print('FNDC5 (4.2%) is low but detectable — consistent with mouse baseline levels.')


---
## Step 5: Per-donor FNDC5 — note the cohort confound

The lean vs. obese comparison is **not interpretable** in this dataset because lean donors
(EPI prefix) and obese donors (TP/UP prefix) come from different hospitals. Site and BMI
are perfectly confounded. We report per-donor values for transparency but make no lean/obese
directional claim.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

sites = {'EPI254':'BIDMC', 'EPI255':'BIDMC', 'EPI266':'BIDMC',
         'EPI253':'BIDMC', 'EPI256':'BIDMC',
         'TP01':'Pittsburgh','UP1005-S':'Pittsburgh','UP1008-B':'Pittsburgh',
         'UP1010-S':'Pittsburgh','UP1018-S':'Pittsburgh'}
site_color = {'BIDMC':'#4878CF', 'Pittsburgh':'#E8604C'}

per_donor = areg.groupby('donor_id').agg(
    bmi=('bmi','first'),
    bmi_group=('bmi__group','first'),
    n=('FNDC5','count'),
    fndc5_mean=('FNDC5','mean'),
    fndc5_pct=('FNDC5', lambda x: (x>0).mean()*100)
).reset_index()
per_donor = per_donor[per_donor['n'] >= 10].sort_values('bmi')
per_donor['site'] = per_donor['donor_id'].map(sites)

colors = [site_color[s] for s in per_donor['site']]
bars = ax.bar(range(len(per_donor)), per_donor['fndc5_mean'], color=colors, alpha=0.8, edgecolor='white')
ax.set_xticks(range(len(per_donor)))
ax.set_xticklabels([f"{row['donor_id']}\nBMI={row['bmi']:.0f}" for _, row in per_donor.iterrows()],
                   fontsize=8, rotation=30, ha='right')
ax.set_ylabel('Mean FNDC5 expression\n(log-normalized counts)')
ax.set_title('FNDC5 in human VAT Areg-equivalent cells (F3+/PDGFRA+/PTPRC-/PECAM1-)
Per donor — lean vs obese comparison confounded by recruitment site')

patches = [mpatches.Patch(color='#4878CF', label='BIDMC (lean/overweight)'),
           mpatches.Patch(color='#E8604C', label='Pittsburgh (obese)')]
ax.legend(handles=patches, fontsize=8)

# Annotation
ax.text(0.02, 0.95, '⚠ Site and BMI group are perfectly confounded
Lean vs. obese comparison not interpretable',
        transform=ax.transAxes, fontsize=8, va='top',
        bbox=dict(boxstyle='round', facecolor='#FFF3CD', alpha=0.8))

plt.tight_layout()
plt.savefig(OUT_DIR / 'fndc5_human_vat_areg_per_donor.png', dpi=150, bbox_inches='tight')
plt.savefig(OUT_DIR / 'fndc5_human_vat_areg_per_donor.pdf', bbox_inches='tight')
plt.show()
print('Figure saved.')


---
## Summary and framing for the paper

### What this notebook establishes

1. **The target cell population exists in human VAT.** F3+/PDGFRA+/PTPRC-/PECAM1- 
   Areg-equivalent cells are present in human omental fat: 1,632 cells out of 80,085 
   total VAT cells (2.0%), predominantly hASPC3.

2. **FNDC5 is detectable at baseline in human VAT Areg-equivalent cells.** 4.2% of 
   cells express it, mean expression 0.045 — comparable to the mouse sedentary baseline 
   (SC mean 0.024) and consistent with the GSE128891 sorted CD142+ baseline (baseMean=25.7).

3. **The CLOCK regulatory machinery is expressed in this population.** NR1D2 (29.5%), 
   TEF (15.6%), PPARGC1B (9.8%), and NR1D1 (1.7%) are all detectable — the same genes 
   that co-induce with FNDC5 in mouse vWAT under exercise.

4. **The lean vs. obese comparison is not interpretable** due to perfect confounding 
   between BMI group and recruitment site. No directional claim is made.

### What this does NOT establish

- Whether FNDC5 expression changes with obesity in human VAT (requires single-site cohort)
- Whether FNDC5 changes with exercise in human VAT (no public dataset has this)
- Whether the mouse vWAT-specificity pattern holds in human VAT (no matched human scWAT 
  from same donors)

### How to frame this in the paper

> No public human dataset combines visceral fat biopsies, exercise conditions, and 
> single-cell resolution — this gap is well-established in the field and motivates 
> the mouse mechanistic work. As a prerequisite check, we confirm that F3+/PDGFRA+ 
> Areg-equivalent cells are detectable in human omental fat at baseline (GSE176171, 
> Emont et al. 2022; n=1,632 of 80,085 VAT cells), and that these cells express FNDC5 
> (4.2% of cells), NR1D2 (29.5%), and PPARGC1B (9.8%) — the same gene module that 
> co-induces in mouse vWAT under exercise training. The human Areg-equivalent 
> population is thus present and transcriptionally primed for the regulatory program 
> identified in mice; direct testing requires an exercise arm that does not currently 
> exist in any public repository.

This framing is honest, adds genuine content (the population exists, the genes are there),
and sets up the wet-lab next step without overclaiming.
